# QRC Demo — Aquila-Style Submission (Python Simulation)

This notebook mirrors the original `QRC Demo Aquila Submission.ipynb` but runs entirely in Python without Julia dependencies. It simulates the **shot-based projective measurement** approach used on QuEra's Aquila quantum computer.

**Key differences from the exact simulation demo:**
- Aquila returns **binary bitstrings** per shot (each atom is either in ground $|g\rangle$ or Rydberg $|r\rangle$).
- We must aggregate many shots to estimate $\langle\sigma_z^i\rangle$ and $\langle\sigma_z^i\sigma_z^j\rangle$.
- We simulate this by sampling from the quantum state probability distribution.

**Pipeline:**
1. Load MNIST and apply PCA (same as QRC MNIST demo).
2. Build the Rydberg Hamiltonian and evolve quantum states.
3. Simulate finite-shot binary measurements.
4. Build embeddings from shot statistics.
5. Train a linear SVM and study how accuracy scales with shot count.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Libraries loaded.")


## Load MNIST and Apply PCA

In [ ]:
try:
    from tensorflow.keras.datasets import mnist
    (X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = mnist.load_data()
    X_tr_raw = X_tr_raw.reshape(-1, 784).astype(np.float32) / 255.0
    X_te_raw = X_te_raw.reshape(-1, 784).astype(np.float32) / 255.0
except ImportError:
    from sklearn.datasets import fetch_openml
    print("Fetching MNIST via OpenML...")
    data = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X_all = data.data.astype(np.float32) / 255.0
    y_all = data.target.astype(int)
    X_tr_raw, X_te_raw = X_all[:60000], X_all[60000:]
    y_tr_raw, y_te_raw = y_all[:60000], y_all[60000:]

NUM_TRAIN, NUM_TEST = 2000, 500   # smaller for speed
X_train_flat, y_train = X_tr_raw[:NUM_TRAIN], y_tr_raw[:NUM_TRAIN]
X_test_flat,  y_test  = X_te_raw[:NUM_TEST],  y_te_raw[:NUM_TEST]

DIM_PCA = 8
pca = PCA(n_components=DIM_PCA)
X_pca_train = pca.fit_transform(X_train_flat)
X_pca_test  = pca.transform(X_test_flat)

DELTA_MAX = 6.0
spectral  = np.max(np.abs(X_pca_train))
X_qrc_train = X_pca_train / spectral * DELTA_MAX
X_qrc_test  = X_pca_test  / spectral * DELTA_MAX
print(f"Train: {X_qrc_train.shape}, Test: {X_qrc_test.shape}")


## Quantum System Setup

We reuse the same Rydberg Hamiltonian as the MNIST notebook.

In [ ]:
sx  = np.array([[0,1],[1,0]], dtype=complex)
sz  = np.array([[1,0],[0,-1]], dtype=complex)
n_op = (np.eye(2,dtype=complex) - sz) / 2

def kron_op(n_qubits, op, site):
    ops = [np.eye(2, dtype=complex)] * n_qubits
    ops[site] = op
    out = ops[0]
    for o in ops[1:]: out = np.kron(out, o)
    return out

def generate_Vmat(locs, C6):
    n = len(locs)
    Vmat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                d = np.linalg.norm(locs[i] - locs[j])
                Vmat[i, j] = C6 / d**6
    return Vmat

def build_rydberg_hamiltonian(nsites, Delta, Omega, Vmat):
    dim = 2**nsites
    H = np.zeros((dim, dim), dtype=complex)
    for i in range(nsites):
        H += (Omega/2) * kron_op(nsites, sx, i)
        H -= Delta[i]  * kron_op(nsites, n_op, i)
    for i in range(nsites):
        for j in range(i+1, nsites):
            if Vmat[i,j] != 0:
                H += Vmat[i,j] * (kron_op(nsites, n_op, i) @ kron_op(nsites, n_op, j))
    return H

NSITES       = 8
ATOM_SPACING = 10.0
C6           = 862690 * 2*np.pi
OMEGA        = 2*np.pi
T_START, T_END, STEP = 0.0, 4.0, 0.5

locs = np.array([[i*ATOM_SPACING, 0.0] for i in range(NSITES)])
Vmat = generate_Vmat(locs, C6)
print("Quantum operators ready.")


## Shot-Based Measurement Simulation

On real Aquila hardware, each "shot" collapses the quantum state to a bitstring.
We simulate this by:
1. Evolving $|\psi\rangle$ under $H$.
2. Computing the full probability distribution $|\langle b|\psi\rangle|^2$ over all bitstrings $b$.
3. Drawing `NUM_SHOTS` samples from this distribution.
4. Estimating $\langle\sigma_z^i\rangle$ and $\langle\sigma_z^i\sigma_z^j\rangle$ from sample frequencies.

$$\hat{\langle\sigma_z^i\rangle} = \frac{1}{S}\sum_{s=1}^{S} (1 - 2\,b_i^{(s)})$$

In [ ]:
def simulate_shots(psi, nsites, num_shots):
    """Draw shot samples from quantum state |psi> and return bitstring array."""    probs = np.abs(psi)**2
    probs /= probs.sum()
    indices = np.random.choice(len(probs), size=num_shots, p=probs)
    # Decode each integer index to a bitstring
    bits = ((indices[:, None] >> np.arange(nsites-1, -1, -1)) & 1).astype(np.int8)
    return bits   # shape (num_shots, nsites)

def shots_to_observables(bits, nsites):
    """Convert binary shots to <Zi> and <ZiZj> estimates."""    z_vals = 1 - 2*bits.astype(float)        # shape (shots, nsites)
    z_mean = z_vals.mean(axis=0)             # <Zi>, shape (nsites,)
    zz_list = []
    for i in range(nsites):
        for j in range(i+1, nsites):
            zz_list.append((z_vals[:,i] * z_vals[:,j]).mean())
    return np.concatenate([z_mean, zz_list]) # length nsites + npairs

def apply_aquila_style_single(Delta, nsites, Vmat, Omega, step, t_end, num_shots):
    """Evolve and collect shot-based embeddings at each readout time."""    H = build_rydberg_hamiltonian(nsites, Delta, Omega, Vmat)
    U = expm(-1j * H * step)
    psi = np.zeros(2**nsites, dtype=complex); psi[0] = 1.0
    steps = int(round(t_end / step))
    parts = []
    for _ in range(steps):
        psi = U @ psi
        bits = simulate_shots(psi, nsites, num_shots)
        parts.append(shots_to_observables(bits, nsites))
    return np.concatenate(parts)

NUM_SHOTS = 200   # ← change this to study shot-count effects

print(f"Using {NUM_SHOTS} shots per readout time.")


## Generating Embeddings (Shot-Based)

In [ ]:
print("Generating shot-based train embeddings...")
emb_train = np.array([
    apply_aquila_style_single(X_qrc_train[i], NSITES, Vmat, OMEGA, STEP, T_END, NUM_SHOTS)
    for i in tqdm(range(len(X_qrc_train)))
])

print("Generating shot-based test embeddings...")
emb_test = np.array([
    apply_aquila_style_single(X_qrc_test[i], NSITES, Vmat, OMEGA, STEP, T_END, NUM_SHOTS)
    for i in tqdm(range(len(X_qrc_test)))
])

print(f"Embeddings shape: {emb_train.shape}")


## Classification

In [ ]:
svm_shots = SVC(kernel='linear', max_iter=5000).fit(emb_train, y_train)
acc_shots = accuracy_score(y_test, svm_shots.predict(emb_test))

svm_pca = SVC(kernel='linear', max_iter=5000).fit(X_qrc_train, y_train)
acc_pca  = accuracy_score(y_test, svm_pca.predict(X_qrc_test))

print(f"PCA + Linear SVM (baseline):          {acc_pca:.3f}")
print(f"Shot-based QRC + Linear SVM ({NUM_SHOTS} shots): {acc_shots:.3f}")


## Effect of Shot Count on Accuracy

More shots → lower statistical noise → closer to the ideal (infinite-shot) expectation value.
Below we sweep `num_shots` to see how accuracy degrades with fewer measurements.

In [ ]:
# Use a smaller subset for speed
SUB = 300
shot_counts = [10, 50, 100, 200, 500, 1000]
accs_by_shots = []

for ns in shot_counts:
    emb_tr_s = np.array([apply_aquila_style_single(X_qrc_train[i], NSITES, Vmat, OMEGA, STEP, T_END, ns)
                         for i in range(SUB)])
    emb_te_s = np.array([apply_aquila_style_single(X_qrc_test[i],  NSITES, Vmat, OMEGA, STEP, T_END, ns)
                         for i in range(SUB)])
    svm_s = SVC(kernel='linear', max_iter=3000).fit(emb_tr_s, y_train[:SUB])
    acc_s = accuracy_score(y_test[:SUB], svm_s.predict(emb_te_s))
    accs_by_shots.append(acc_s)
    print(f"{ns:5d} shots → Test Acc = {acc_s:.3f}")

plt.figure(figsize=(7, 4))
plt.semilogx(shot_counts, accs_by_shots, 'o-', color='darkorchid', linewidth=2)
plt.axhline(acc_pca, linestyle='--', color='gray', label=f'PCA baseline ({acc_pca:.3f})')
plt.xlabel("Number of Shots"); plt.ylabel("Test Accuracy")
plt.title("QRC Accuracy vs. Shot Count")
plt.legend(); plt.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()


## Summary

| Method | Test Accuracy |
|--------|--------------|
| PCA + Linear SVM (baseline) | — |
| QRC + Linear SVM (finite shots) | — |

**Key observations:**
- Even with moderate shot counts (~200), the QRC embedding is useful for classification.
- Accuracy saturates as shot count increases, approaching the exact quantum simulation result.
- On real Aquila hardware, shot counts of 100–1000 are typical, balancing accuracy with runtime.

**Next steps:**
- Use the `bloqade-python` SDK to submit actual jobs to Aquila (replace the `simulate_shots` function with hardware calls).
- Explore atom position modulation as an alternative encoding strategy.